<a href="https://colab.research.google.com/github/Pensive1881/DSR44_2025/blob/main/jaguar_image_segmentation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Install FiftyOne

In [1]:
%%capture
!uv pip install fiftyone==1.9.0

In [2]:
import os
from google.colab import userdata
FIFTYONE_CVAT_PASSWORD = userdata.get('FIFTYONE_CVAT_PASSWORD')
FIFTYONE_CVAT_USERNAME = userdata.get('FIFTYONE_CVAT_USERNAME')

In [3]:
!export FIFTYONE_CVAT_USERNAME={FIFTYONE_CVAT_USERNAME}
!export FIFTYONE_CVAT_PASSWORD={FIFTYONE_CVAT_PASSWORD}

## Mount Google Drive

In [4]:
from google.colab import drive
drive.mount('/gdrive')
%cd /gdrive

Mounted at /gdrive
/gdrive


In [5]:
from pathlib import Path
train_set_files = Path('/gdrive/MyDrive/train_set_parquet_files')

os.listdir(train_set_files)

['cropped_body-00000-of-00005.parquet',
 'cropped_body-00001-of-00005.parquet',
 'cropped_body-00002-of-00005.parquet',
 'cropped_body-00003-of-00005.parquet',
 'cropped_body-00004-of-00005.parquet',
 'segmented_body-00000-of-00003.parquet',
 'segmented_body-00001-of-00003.parquet',
 'segmented_body-00002-of-00003.parquet',
 'cropped_head-00000-of-00002.parquet',
 'cropped_head-00001-of-00002.parquet']

## Create the FiftyOne dataset

In [6]:
"""## Install additional dependencies"""

# Commented out IPython magic to ensure Python compatibility.
# %%capture
# !uv pip install pyarrow pillow tqdm

"""## Extract images from Parquet files"""

import pyarrow.parquet as pq
from PIL import Image
from io import BytesIO
from tqdm import tqdm
import fiftyone as fo

def extract_images_from_parquet(parquet_dir, image_type=None, output_dir="fiftyone_images"):
    """
    Extract images from Parquet files and save to disk.

    Args:
        parquet_dir: Directory containing Parquet files
        image_type: Specific image type to extract (cropped_body, cropped_head, segmented_body) or None for all
        output_dir: Directory to save extracted images

    Returns:
        List of tuples: (image_path, label, image_type, original_filename)
    """
    output_path = Path(output_dir)
    output_path.mkdir(exist_ok=True)

    # Get relevant parquet files
    parquet_files = sorted(Path(parquet_dir).glob("*.parquet"))

    if image_type:
        parquet_files = [f for f in parquet_files if f.name.startswith(image_type)]
        if not parquet_files:
            raise ValueError(f"No files found for image type: {image_type}")

    print(f"Found {len(parquet_files)} parquet files to process")

    samples_data = []

    for parquet_file in parquet_files:
        # Extract image type from filename (e.g., "cropped_body-00000-of-00005.parquet")
        current_image_type = parquet_file.name.split('-')[0]

        print(f"\nProcessing: {parquet_file.name}")
        table = pq.read_table(parquet_file)

        # Create subdirectory for this image type
        type_dir = output_path / current_image_type
        type_dir.mkdir(exist_ok=True)

        # Extract data
        filenames = table.column('filename').to_pylist()
        labels = table.column('label').to_pylist()
        images = table.column('image').to_pylist()

        for idx, (filename, label, img_data) in enumerate(tqdm(
            zip(filenames, labels, images),
            total=len(filenames),
            desc=f"Extracting {current_image_type}"
        )):
            if img_data and img_data['bytes']:
                # Create unique filename: label_originalname_imagetype
                base_name = Path(filename).stem
                ext = Path(filename).suffix or '.jpg'
                unique_name = f"{label}_{base_name}_{current_image_type}{ext}"
                img_path = type_dir / unique_name

                # Save image
                try:
                    img = Image.open(BytesIO(img_data['bytes']))
                    img.save(img_path)

                    samples_data.append({
                        'filepath': str(img_path.absolute()),
                        'label': label,
                        'image_type': current_image_type,
                        'original_filename': filename
                    })
                except Exception as e:
                    print(f"Warning: Failed to process {filename}: {e}")

    return samples_data

"""## Create FiftyOne Dataset"""

def create_fiftyone_dataset(samples_data, dataset_name="jaguar_reid", persistent=True):
    """
    Create a FiftyOne dataset from extracted samples.

    Args:
        samples_data: List of sample dictionaries with filepath, label, image_type, original_filename
        dataset_name: Name for the FiftyOne dataset
        persistent: Whether to make the dataset persistent

    Returns:
        FiftyOne Dataset object
    """
    # Delete existing dataset with same name
    if dataset_name in fo.list_datasets():
        print(f"Deleting existing dataset: {dataset_name}")
        fo.delete_dataset(dataset_name)

    # Create new dataset
    dataset = fo.Dataset(dataset_name, persistent=persistent)

    print(f"\nCreating FiftyOne dataset: {dataset_name}")

    # Create samples
    samples = []
    for data in tqdm(samples_data, desc="Creating FiftyOne samples"):
        sample = fo.Sample(filepath=data['filepath'])

        # Add classification label
        sample['ground_truth'] = fo.Classification(label=data['label'])

        # Add custom fields
        sample['image_type'] = data['image_type']
        sample['original_filename'] = data['original_filename']

        samples.append(sample)

    # Add samples to dataset
    dataset.add_samples(samples)

    print(f"\nDataset created successfully!")
    print(f"  Name: {dataset_name}")
    print(f"  Total samples: {len(dataset)}")
    print(f"  Unique labels: {len(dataset.distinct('ground_truth.label'))}")
    print(f"  Image types: {dataset.distinct('image_type')}")

    return dataset

"""## Run the pipeline"""

# Configuration
PARQUET_DIR = train_set_files
OUTPUT_DIR = "/gdrive/MyDrive/fiftyone_images"  # Save in Google Drive
DATASET_NAME = "jaguar_reid"
IMAGE_TYPE = None  # None for all types, or specify: 'cropped_body', 'cropped_head', 'segmented_body'

# Step 1: Extract images from Parquet files
print("Step 1: Extracting images from Parquet files...")
samples_data = extract_images_from_parquet(
    PARQUET_DIR,
    image_type=IMAGE_TYPE,
    output_dir=OUTPUT_DIR
)

print(f"\nExtracted {len(samples_data)} images")

# Step 2: Create FiftyOne dataset
print("\nStep 2: Creating FiftyOne dataset...")
dataset = create_fiftyone_dataset(
    samples_data,
    dataset_name=DATASET_NAME,
    persistent=True
)

# Print summary statistics
print("\n" + "="*60)
print("Dataset Summary:")
print("="*60)
print(dataset)
print("\nLabel distribution:")
for label in sorted(dataset.distinct('ground_truth.label')):
    count = len(dataset.match(fo.ViewField('ground_truth.label') == label))
    print(f"  {label}: {count} samples")

"""## Launch FiftyOne App

To view the dataset in Google Colab, use the remote session:
"""

# Launch FiftyOne app with remote session for Colab
session = fo.launch_app(dataset, auto=False)

# The session will provide a link you can click to view the dataset
print("\nClick the link above to view your dataset in FiftyOne!")

print(session.url)

/usr/local/lib/python3.12/dist-packages/glob2/fnmatch.py:141: SyntaxWarning: invalid escape sequence '\Z'
  return '(?ms)' + res + '\Z'


Step 1: Extracting images from Parquet files...
Found 10 parquet files to process

Processing: cropped_body-00000-of-00005.parquet


Extracting cropped_body: 100%|██████████| 620/620 [08:06<00:00,  1.27it/s]



Processing: cropped_body-00001-of-00005.parquet


Extracting cropped_body: 100%|██████████| 620/620 [07:17<00:00,  1.42it/s]


...

INFO:fiftyone.core.session.session:Session launched. Run `session.show()` to open the App in a cell output.



Welcome to

███████╗██╗███████╗████████╗██╗   ██╗ ██████╗ ███╗   ██╗███████╗
██╔════╝██║██╔════╝╚══██╔══╝╚██╗ ██╔╝██╔═══██╗████╗  ██║██╔════╝
█████╗  ██║█████╗     ██║    ╚████╔╝ ██║   ██║██╔██╗ ██║█████╗
██╔══╝  ██║██╔══╝     ██║     ╚██╔╝  ██║   ██║██║╚██╗██║██╔══╝
██║     ██║██║        ██║      ██║   ╚██████╔╝██║ ╚████║███████╗
╚═╝     ╚═╝╚═╝        ╚═╝      ╚═╝    ╚═════╝ ╚═╝  ╚═══╝╚══════╝ v1.9.0

If you're finding FiftyOne helpful, here's how you can get involved:

|
|  ⭐⭐⭐ Give the project a star on GitHub ⭐⭐⭐
|  https://github.com/voxel51/fiftyone
|
|  🚀🚀🚀 Join the FiftyOne Discord community 🚀🚀🚀
|  https://community.voxel51.com/
|



INFO:fiftyone.core.session.session:
Welcome to

███████╗██╗███████╗████████╗██╗   ██╗ ██████╗ ███╗   ██╗███████╗
██╔════╝██║██╔════╝╚══██╔══╝╚██╗ ██╔╝██╔═══██╗████╗  ██║██╔════╝
█████╗  ██║█████╗     ██║    ╚████╔╝ ██║   ██║██╔██╗ ██║█████╗
██╔══╝  ██║██╔══╝     ██║     ╚██╔╝  ██║   ██║██║╚██╗██║██╔══╝
██║     ██║██║        ██║      ██║   ╚██████╔╝██║ ╚████║███████╗
╚═╝     ╚═╝╚═╝        ╚═╝      ╚═╝    ╚═════╝ ╚═╝  ╚═══╝╚══════╝ v1.9.0

If you're finding FiftyOne helpful, here's how you can get involved:

|
|  ⭐⭐⭐ Give the project a star on GitHub ⭐⭐⭐
|  https://github.com/voxel51/fiftyone
|
|  🚀🚀🚀 Join the FiftyOne Discord community 🚀🚀🚀
|  https://community.voxel51.com/
|




Click the link above to view your dataset in FiftyOne!


KeyboardInterrupt: 

## Start a semantic segmentation annotation job on CVAT

In [7]:
"""## Start a semantic segmentation annotation job on CVAT

For semantic segmentation, we need to:
1. Set up mask targets (mapping pixel values to class labels)
2. Configure the annotation schema
3. Upload the dataset to CVAT for annotation
"""

# Define mask targets: mapping pixel values to semantic label strings
# We only care about two classes: body and background
mask_targets = {
    0: "background",  # Pixel value 0 = background
    1: "body",        # Pixel value 1 = body
}

# Store mask targets on the dataset
# This allows FiftyOne to automatically use them when creating the annotation task
dataset.default_mask_targets = mask_targets
dataset.save()

print("\nMask targets configured:")
print(f"  0: background")
print(f"  1: body")

"""### Configure and launch the annotation task

We'll create a new field called 'segmentation' to store the semantic segmentation masks.
"""

# Unique identifier for this annotation run
anno_key = "jaguar_segmentation"

# Configure the annotation task
# For semantic segmentation with CVAT, we specify:
# - label_field: the field where segmentation masks will be stored
# - label_type: "segmentation" for semantic segmentation
# - mask_targets: the mapping between pixel values and class labels
# - launch_editor: whether to open CVAT in the browser

print(f"\nCreating CVAT annotation task '{anno_key}'...")
print(f"Field: segmentation")
print(f"Type: semantic segmentation")
print(f"Classes: {list(mask_targets.values())}")

results = dataset.annotate(
    anno_key,
    label_field="segmentation",
    label_type="segmentation",
    mask_targets=mask_targets,
    launch_editor=False,  # Set to True to automatically open CVAT
)

# Print information about the created annotation task
print("\nAnnotation task created successfully!")
print(dataset.get_annotation_info(anno_key))

"""### View task information"""

# Get the status of the annotation task
print("\nTask Status:")
results.print_status()

# Get the CVAT task URL
print("\nTo annotate in CVAT, visit the task URL shown above.")
print("Or use the CVAT UI at: https://app.cvat.ai")

"""### After annotation is complete, load the results

Once you've completed the annotation work in CVAT, run the following code
to download and merge the annotations back into your FiftyOne dataset:
"""

# Uncomment the following lines after completing annotation in CVAT:

# # Load annotations from CVAT back into FiftyOne
# dataset.load_annotations(anno_key)
#
# # View the annotated samples
# session = fo.launch_app(dataset)
#
# # Inspect a sample with its segmentation mask
# sample = dataset.first()
# print(sample.segmentation)

"""### Optional: Cleanup after annotation

If you want to delete the CVAT tasks and remove the annotation run record
from your FiftyOne dataset, use:
"""

# Uncomment to cleanup:
# # Delete tasks from CVAT
# results.cleanup()
#
# # Delete run record (not the labels) from FiftyOne
# dataset.delete_annotation_run(anno_key)



Mask targets configured:
  0: background
  1: body

Creating CVAT annotation task 'jaguar_segmentation'...
Field: segmentation
Type: semantic segmentation
Classes: ['background', 'body']
Please enter your login credentials.
You can avoid this in the future by setting your `FIFTYONE_CVAT_USERNAME` and `FIFTYONE_CVAT_PASSWORD` environment variables


INFO:fiftyone.utils.annotations:Please enter your login credentials.
You can avoid this in the future by setting your `FIFTYONE_CVAT_USERNAME` and `FIFTYONE_CVAT_PASSWORD` environment variables


Username: hungrymagic
Password: ··········
Computing metadata...


INFO:fiftyone.core.metadata:Computing metadata...


 100% |███████████████| 9294/9294 [1.2m elapsed, 0s remaining, 128.4 samples/s]      


INFO:eta.core.utils: 100% |███████████████| 9294/9294 [1.2m elapsed, 0s remaining, 128.4 samples/s]      


By default, all images are uploaded to CVAT in a single task, but this requires loading all images simultaneously into RAM, which will take at least 4.2GB. Consider specifying a `task_size` to break the data into smaller chunks, or upgrade to FiftyOne Enterprise so that you can provide a cloud manifest


Uploading samples to CVAT...


INFO:fiftyone.utils.cvat:Uploading samples to CVAT...


SSLError: HTTPSConnectionPool(host='app.cvat.ai', port=443): Max retries exceeded with url: /api/tasks/1748505/data (Caused by SSLError(SSLEOFError(8, 'EOF occurred in violation of protocol (_ssl.c:2427)')))